In [ ]:
# Install necessary libraries (you only need to run this once)
!pip install erddapy pandas numpy scikit-learn matplotlib

In [16]:


# Import core data engineering tools
from erddapy import ERDDAP      # Specifically for fetching NOAA data [cite: 16, 42]
import pandas as pd             # For time-series cleaning and data tables [cite: 17, 59]
import numpy as np              # For numerical calculations and weights [cite: 33]
import matplotlib.pyplot as plt # For immediate visual feedback/plotting [cite: 52, 60]

print("Environment setup complete. Ready to fetch NOAA data!")

Environment setup complete. Ready to fetch NOAA data!


In [3]:
pip install ndbc-api

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [17]:
from ndbc_api import NdbcApi

api = NdbcApi()

# 2. Define your "Virtual Buoy" target location (Example: Monterey Bay area)
# Use 'N/S' and 'E/W' string formatting required by the API
VIRTUAL_LAT = '32.693442805149374N'
VIRTUAL_LON = '-117.25328402161438E'
SEARCH_RADIUS_KM = 100

print(f"Searching for physical NOAA buoys within {SEARCH_RADIUS_KM}km of ({VIRTUAL_LAT}, {VIRTUAL_LON})...")

# 3. Perform a radial search to find neighboring physical stations
nearby_stations = api.radial_search(
    lat=VIRTUAL_LAT, 
    lon=VIRTUAL_LON, 
    radius=SEARCH_RADIUS_KM, 
    units='km'
)

# Display the nearest stations found
nearby_stations.head()

Searching for physical NOAA buoys within 100km of (32.693442805149374N, -117.25328402161438E)...


,Station,Lat,Lon,Elevation,Name,Owner,Program,Type,Includes Meteorology,Includes Currents,Includes Water Quality,DART Program,distance
1210,sdbc1,32.714,-117.174,NaN,"9410170 - San Diego, CA",NOS,NOS/CO-OPS,fixed,True,False,False,False,7.771150
434,46235,32.570,-117.169,NaN,"Imperial Beach Nearshore, CA (155)",SCRIPPS,IOOS Partners,buoy,False,False,False,False,15.850984
1068,npqc1,32.601,-117.116,NaN,"South Bay, Tijuana River Reserve, CA",National Estuarine Research Reserve System,NERRS,fixed,False,False,True,False,16.476320
1283,tixc1,32.575,-117.127,4.0,"Tidal Linkage, Tijuana River Reserve, CA",National Estuarine Research Reserve System,NERRS,fixed,True,False,False,False,17.719537
1282,tiqc1,32.568,-117.131,NaN,"Oneonta Slough, Tijuana River Reserve, CA",National Estuarine Research Reserve System,NERRS,fixed,False,False,True,False,18.066807


In [34]:
# Create a dictionary to hold dataframes for each buoy
buoy_data_dict = {}

# We will loop through all 10 nearby stations to build our virtual buoy network
target_buoys = nearby_stations['Station'].head(10).tolist()

# Define the modes we want to cycle through as fallbacks
MODES_TO_TRY = ['stdmet', 'cwind', 'ocean', 'spec']

print(f"Ingesting data using multi-mode fallback for buoys: {target_buoys}\n")

for station_id in target_buoys:
    data_found = False
    
    # Try each mode sequentially until one yields rows
    for mode in MODES_TO_TRY:
        try:
            data_response = api.get_data(station_id=station_id, mode=mode)
            
            # Standard dictionary-to-dataframe conversion
            if isinstance(data_response, dict):
                raw_df = pd.DataFrame.from_dict(data_response)
            else:
                raw_df = data_response
                
            if raw_df is None or raw_df.empty:
                continue
                
            # Standardize column headers to UPPERCASE
            raw_df.columns = [col.upper() for col in raw_df.columns]
            available_cols = raw_df.columns.tolist()
            
            # Map standard NOAA columns or alternative column names if they exist
            # Note: 'WTMP' is water temp, which can serve as a great backup feature!
            features_to_keep = [col for col in ['WVHT', 'WSPD', 'WTMP'] if col in available_cols]
            
            if not features_to_keep:
                continue
                
            # Keep rows that have at least ONE valid feature measurement
            cleaned_df = raw_df[features_to_keep].copy().dropna(how='all')
            
            if not cleaned_df.empty:
                buoy_data_dict[station_id] = cleaned_df
                print(f" ✅ [Success] Station {station_id} loaded using mode '{mode}'. Rows: {len(cleaned_df)}")
                print(f"    Features captured: {features_to_keep}")
                data_found = True
                break # Break out of the mode loop for this buoy since we found data!
                
        except Exception:
            continue # If a mode completely crashes, silently try the next mode
            
    if not data_found:
        print(f" ❌ [No Data] Station {station_id} returned no metrics across all modes.")

# Verify our final multi-buoy dataset distribution
print("\n--- Final Ingestion Summary ---")
print(f"Active framework nodes available for modeling: {list(buoy_data_dict.keys())}")

Ingesting data using multi-mode fallback for buoys: ['sdbc1', '46235', 'npqc1', 'tixc1', 'tiqc1', 'ljac1', 'ljpc1', '46254', '46258', '46232']

 ✅ [Success] Station sdbc1 loaded using mode 'stdmet'. Rows: 6242
    Features captured: ['WVHT', 'WSPD', 'WTMP']
 ❌ [No Data] Station 46235 returned no metrics across all modes.
 ❌ [No Data] Station npqc1 returned no metrics across all modes.
 ✅ [Success] Station tixc1 loaded using mode 'stdmet'. Rows: 2870
    Features captured: ['WVHT', 'WSPD', 'WTMP']
 ❌ [No Data] Station tiqc1 returned no metrics across all modes.
 ✅ [Success] Station ljac1 loaded using mode 'stdmet'. Rows: 6868
    Features captured: ['WVHT', 'WSPD', 'WTMP']
 ✅ [Success] Station ljpc1 loaded using mode 'stdmet'. Rows: 719
    Features captured: ['WVHT', 'WSPD', 'WTMP']
 ✅ [Success] Station 46254 loaded using mode 'stdmet'. Rows: 1438
    Features captured: ['WVHT', 'WSPD', 'WTMP']
 ✅ [Success] Station 46258 loaded using mode 'stdmet'. Rows: 1439
    Features captured: ['W

In [35]:
import os

# 1. Create a local 'data' directory if it doesn't exist
os.makedirs("../data", exist_ok=True)

print("Exporting cleaned time-series tables to local storage...")

# 2. Iterate through your successful nodes and save them as separate CSV files
for station_id, dataframe in buoy_data_dict.items():
    output_path = f"../data/{station_id}_cleaned.csv"
    dataframe.to_csv(output_path)
    print(f" Saved data for station {station_id} -> {output_path} ({len(dataframe)} rows)")

print("\n🎉 Phase I: Data Ingestion Notebook Complete!")

Exporting cleaned time-series tables to local storage...
 Saved data for station sdbc1 -> ../data/sdbc1_cleaned.csv (6242 rows)
 Saved data for station tixc1 -> ../data/tixc1_cleaned.csv (2870 rows)
 Saved data for station ljac1 -> ../data/ljac1_cleaned.csv (6868 rows)
 Saved data for station ljpc1 -> ../data/ljpc1_cleaned.csv (719 rows)
 Saved data for station 46254 -> ../data/46254_cleaned.csv (1438 rows)
 Saved data for station 46258 -> ../data/46258_cleaned.csv (1439 rows)
 Saved data for station 46232 -> ../data/46232_cleaned.csv (1438 rows)

🎉 Phase I: Data Ingestion Notebook Complete!
